# Private Federated Aggregation of LLM-Agent Memory
### Usable Shared Memory with a Measurable Leakage-Drop Guarantee under Secure Aggregation and the Skellam Mechanism

*Draft — v0.2 (2026-07-04). Empirical results are the 5-seed, final-metric rerun of the oracle/small grid. The full 247k-note `s`-scale generality check is now **complete** (§7.7) and the LLM-distilled-payload results (§7.4) are complete. Only the **full LLM-driven MEXTRA/MRMMIA** attacks — which would upgrade the embedding-similarity MIA proxy used here to the published attacks — remain in progress and are marked **[PENDING]**.*

---

## Abstract

LLM agents increasingly accumulate **per-user memory** — distilled notes, preferences, and
skill embeddings (as in A-MEM and Mem0) — that make them personal and effective. Pooling these
memories *across users* would let a fleet of agents share hard-won knowledge, but the memory
objects are exactly the sensitive artifact: recent extraction (MEXTRA) and membership-inference
(MRMMIA) attacks recover verbatim user facts from agent memory. We ask whether a shared memory
pool can be made **useful and provably private at once**.

We aggregate per-user memory-note embeddings into a shared, bucketed **centroid memory** under
**Secure Aggregation (SecAgg)** composed with the **discrete Skellam mechanism**, so the pooled
memory carries a single central differential-privacy guarantee with **no trusted curator** and
**no individual note in the clear**. The payload is new but the cryptographic path is verbatim
prior work, which lets us reuse a rigorous discrete-DP accountant. We evaluate on **LongMemEval**
(real long-horizon conversational memory) and on **LLM-distilled A-MEM/Mem0-style notes**, along
two axes: (i) **retrieval utility** of the noised pool, and (ii) the **drop in worst-case
re-identification** of vulnerable members.

Three findings. **(1) Usable DP memory.** In the regime where agent payloads actually live —
**tiny embedding dimension and coarse bucketing** — the private pool retains **96–97% of clean
retrieval utility at ε≈9.3** on real data. **(2) A measurable leakage-drop endpoint.**
Membership-inference AUC on the vulnerable low-count tail falls from **0.88–0.99 (near-certain)
to ≈0.51 (chance) at ε≈9.3**, while bulk utility holds — the *positive, mechanism-grounded*
guarantee that generic federated-DP results lack. **(3) The dimension/density crossover governs
both axes at once.** Higher embedding dimension *simultaneously* increases pre-DP leakage and
decreases DP utility-retention, so **tiny-d is Pareto-preferred** — precisely where Skellam's
low-precision advantage compounds. We further show the discrete mechanism is **privacy-free at
our quantiser resolution** and that a **vector-only release strictly dominates** the natural
two-channel design (identical utility, ~1.5× tighter ε). Counter-intuitively, **LLM-distilled
notes leak *more* than raw turns** (distillation concentrates identifying facts) — yet DP
collapses both to chance, so the realistic payload *strengthens* the case. An at-scale run on the
full 247k-note corpus (§7.7) confirms the honest boundary of the headline: in the dense regime the
vulnerable tail vanishes and DP is nearly free on both axes.

## 1. Introduction

**Agent memory is the new sensitive payload.** Production LLM-agent frameworks (A-MEM, Mem0)
persist a per-user store of distilled notes — "the user is allergic to penicillin," "prefers
metric units," "is migrating a Postgres 14 cluster." This memory is what makes an agent
personal, and it is also a dense concentrate of personally identifying information. A natural,
high-value idea is to **share memory across users**: if one user's agent learned a robust fix,
every agent should benefit. But naïvely pooling raw notes is a privacy disaster, and recent work
shows the threat is concrete, not hypothetical — **MEXTRA** [11] extracts verbatim memory
content and **MRMMIA** [12] performs membership inference against agent memory with high success.

**Why the obvious defenses are insufficient.** Access-control layers (*Collaborative Memory* [13])
and on-device masking (*MemPrivacy* [14]) restrict *who* sees a note but provide no formal
guarantee over the *aggregate*, and a curated central datastore [15] reintroduces a trusted
party. What is missing is a mechanism that (a) needs **no trusted curator**, (b) gives a
**central-DP guarantee over the shared pool**, and (c) is shown to actually **degrade the known
attacks** while keeping the pool useful.

**Our approach.** We treat federated memory aggregation as a **secure summation of per-user
bucketed embeddings**. Each user maps their notes into `K` shared buckets (random-hyperplane
LSH), forms a per-bucket sum-vector, and contributes it through **SecAgg** [2] so the server sees
only the masked sum; the **Skellam mechanism** [5] adds calibrated discrete noise that
**composes under summation**, yielding a single distributed-DP guarantee on the pooled centroid
memory. Retrieval is cosine lookup against the noised centroids. The cryptographic path is used
**verbatim** from federated discrete-DP work; the novelty is the *payload* (agent-memory
embeddings) and the *evaluation* (a measured attack-success drop).

**Why this payload, and why now.** Federated DP for LLM *adaptation* (DP-LoRA, prompt tuning) is
saturated, and preference-vector aggregation is near-scooped by POPri [16] (continuous Gaussian,
no Skellam). Agent *memory* aggregation under SecAgg+discrete-DP is an open seam — a 2026 survey
[18] explicitly names "DP in federated agent adaptation" as a gap, and Fed-SE [17] lists DP as
future work. Crucially, agent-memory objects are **genuinely tiny-dimensional and often already
discrete**, which is exactly the regime where the discrete Skellam mechanism is most efficient
and where our dimension analysis is most credible.

**Contributions.**
1. **A federated agent-memory aggregation mechanism** that composes SecAgg with the Skellam
   mechanism over bucketed memory-note embeddings, giving a curator-free central-DP shared
   memory pool (§4–5).
2. **A positive, measurable privacy endpoint**: on real conversational memory the worst-case
   membership-inference AUC over vulnerable (low-count) members drops from **0.88–0.99 to
   ≈0.51 at ε≈9.3**, while retrieval utility is retained (§7.3).
3. **The dimension/density crossover as a design principle**: higher `d` raises pre-DP leakage
   *and* lowers DP utility-retention, making **tiny-d Pareto-optimal**; we quantify it on real
   sentence embeddings (§7.2).
4. **A rigorous, tightened accounting**: an exact Skellam-RDP ε for the discrete mechanism
   (Agarwal et al. Thm 3.5 [5]), a proof that **discretisation is free** at our resolution, and
   a **vector-only release** that strictly dominates the two-channel design (§5.3, §7.5).
5. **A realistic-payload validation**: LLM-distilled A-MEM/Mem0-style notes leak *more* than raw
   turns, yet DP still collapses the attack to chance — the real payload strengthens, not
   weakens, the result (§7.4).

All results are reported as **5-seed means with final (not peak) metrics**; the harness and grid
are released (§10).

## 2. Related Work

**Federated DP with secure aggregation and discrete noise.** SecAgg [2] lets a server compute a
sum of client vectors without seeing any summand. To obtain a DP guarantee that composes under
that sum without a trusted curator, discrete mechanisms are used: the **distributed discrete
Gaussian** [6] and the **Skellam mechanism** [5], the latter being closed under addition (the sum
of Skellams is Skellam) and efficient at low bit-width. Prior deployments target **gradient**
payloads for model training. We reuse this path *verbatim* but change the payload to **agent-memory
embeddings** and the objective to **retrieval**, not learning.

**Privacy of LLM-agent memory.** Agent frameworks A-MEM [9] and Mem0 [10] persist distilled
per-user memories. Their privacy is under active attack: **MEXTRA** [11] extracts memory content
and **MRMMIA** [12] runs membership inference; *AgentLeak* and related probes round out a
three-deep measurement literature. Defenses so far are **access control** (*Collaborative
Memory* [13]) and **on-device masking** (*MemPrivacy* [14]) — neither gives an aggregate formal
guarantee. We therefore **cite and reuse the attacks as our evaluation harness** rather than
claiming them, and provide the missing mechanism-side guarantee.

**Federated retrieval / datastores.** *DP Datastore Generation* [15] privatizes a retrieval
datastore but is **centralized, single-store, with plain additive noise and no SecAgg**; our
federated, SecAgg-composed, summation-closed framing is a clean delta. **POPri** [16] aggregates
*preference* vectors under SecAgg + user-level DP but uses **continuous Gaussian, not Skellam**,
and targets preferences, not memory retrieval.

**Dimension dependence of DP.** That utility degrades with dimension under DP is classical
(Bassily–Smith–Thakurta [20]) and was sharpened for deep models by Chen et al. [19]. We do **not**
claim the dimension dependence itself; our contribution is the **coupled crossover** — that in
agent memory, dimension governs pre-DP *leakage* and DP *utility-retention simultaneously*, making
tiny-d a joint optimum — and the demonstration on real memory embeddings.

## 3. Threat Model and Problem Setup

**Parties.** `N` users, each with a local agent holding a set of memory notes; an honest-but-curious
aggregation server; and downstream agents that query the shared memory.

**Goal.** Produce a single **shared memory pool** — a set of `K` centroid embeddings — that any
agent can query by cosine similarity to retrieve relevant knowledge contributed by the fleet,
such that the pool carries a **central (ε, δ)-DP guarantee at user granularity** and no
individual user's notes can be reconstructed or their membership inferred.

**Trust / adversary.** No trusted curator: under SecAgg the server observes only the masked sum,
so the DP noise need not be added by a trusted party. The adversary is a recipient of the
released pool (server or any downstream agent). It mounts:
- **Membership inference (MRMMIA analog):** given a candidate note embedding `x`, decide whether
  the user who owns `x` contributed to the pool. Score `s(x) = cos(x, centroid[bucket(x)])`;
  members pulled their own centroid and score higher. Metric: ROC-AUC (0.5 = no leakage).
- **Extraction (MEXTRA analog):** advantage in reconstructing an in-bucket member embedding from
  the centroid; we report the member/non-member **extraction gap**.

**The vulnerable subpopulation.** In a *sum* mechanism, leakage concentrates where **few users
contribute to a bucket** — averaging already protects well-populated buckets, and distributed-DP
noise is *most* protective exactly at the sparse tail. We therefore stratify every leakage metric
by the **low-count tail** (buckets with ≤ 3 contributors), which is the honest worst case and the
group agent-memory privacy actually cares about (rare/unique notes).

**Utility.** For a query `q`, retrieval routes to the top-`k` centroids by cosine; utility is
whether the *right* content is retrieved (evidence-recall / answer-recall on real benchmarks,
topic-accuracy on synthetic corpora).

## 4. Bucketed Centroid Memory

Let each note `i` of user `u` have embedding `e_i ∈ R^D` from a sentence encoder. We reduce to a
working dimension `d` (PCA), L2-normalize, and assign to one of `K` buckets by
random-hyperplane LSH: fix `K` random anchors `a_1..a_K` and set `bucket(i) = argmax_k ⟨e_i, a_k⟩`.
Each user forms, per bucket `k`, a **sum-vector** `v_u[k] = Σ_{i: bucket(i)=k} e_i` and a
**count** `c_u[k]`. The (non-private) shared centroid is

$$ \mu[k] \;=\; \mathrm{normalize}\!\Big( \tfrac{\sum_u v_u[k]}{\sum_u c_u[k]} \Big). $$

Retrieval for query `q` returns the top-`k` buckets by `cos(q, \mu[k])`. `K` controls memory
*fidelity* (more buckets = finer memory), `d` controls embedding *resolution*; both, as we show,
trade off against privacy.

## 5. Private Release: SecAgg + Skellam

**Per-user clipping and quantisation.** Each `v_u` is clipped to an L2 bound `C` (the 95th
percentile of user norms) and quantised to integers on a fixed grid of resolution
`s = range_max / B` (we use `range_max = 1e6`). This yields integer vectors amenable to SecAgg
and to a discrete noise mechanism.

**Skellam noise, composed under the sum.** Each user adds Skellam noise (difference of two
Poissons, per-coordinate variance `μ = (σ C s)^2`) to their quantised vector and submits it
through **SecAgg**, so the server recovers only `Σ_u (quantise(clip(v_u)) + Skellam)`. Because the
sum of independent Skellams is Skellam, the *released sum* carries the target noise with **no
trusted curator**. Dequantising and dividing by the (also-released or cancelled, §5.3) count gives
the private centroid `\tilde\mu`.

### 5.3 Vector-only release (the free win)

The natural design releases **two** channels — the sum-vector and the counts — costing two RDP
compositions. But the retrieval centroid is `normalize(sum_v / count)`, and dividing by the scalar
count then L2-normalising **cancels the count**: `normalize(sv/sc) = normalize(sv)`. The count
never affects retrieval direction. Hence we release **only the summed vector** (one composition).
This is **strictly dominant**: identical retrieval to the last decimal (same seeded sum-vector
noise) at **~1.5× tighter ε** (§7.5). We call this the *vector-only* release and adopt it as the
recommended mechanism. (If bucket occupancy must itself be privatised, add a cheap separate
low-sensitivity count release; for pure retrieval it is unnecessary.)

### 5.4 Privacy accounting (exact Skellam-RDP)

We account with the exact discrete guarantee of Agarwal, Kairouz & Liu (NeurIPS 2021, Thm 3.5) [5]:

$$ \varepsilon_{\mathrm{RDP}}(\alpha) \le \frac{\alpha \Delta_2^2}{2\mu} + \min\!\Big\{ \frac{(2\alpha-1)\Delta_2^2 + 6\Delta_1}{4\mu^2},\; \frac{3\Delta_1}{2\mu} \Big\}, $$

with `μ` the per-coordinate released variance, `Δ_2 = C s`, `Δ_1 ≤ √d · Δ_2`. Composition over
releases is additive in RDP; we convert to (ε, δ) by `ε = min_α [RDP(α) + ln(1/δ)/(α−1)]`
(δ = 1e-5). This is implemented as `skellam_rdp_epsilon(...)` in `qpriviot_fl/privacy_utils.py`
and replaces the approximate single-shot Gaussian labels used during exploration. **Every ε in
this paper is the rigorous Skellam-RDP value.** A representative mapping at the K=32, d=32
operating point:

| σ | classic (invalid ε>1) | **Skellam, vector-only (1 release)** | Skellam, two-channel (2 releases) |
|---|---|---|---|
| 0.303 | 15.99 | 22.1 | 33.3 |
| 0.606 | 7.99 | **9.28** | 13.93 |
| 1.615 | 3.00 | **3.16** | 4.60 |
| 2.854 | 1.70 | **1.74** | 2.50 |

The exploration labels "ε=8 / ε=3" correspond to the rigorous **ε≈9.3 / ε≈3.2** under the
recommended vector-only release; we use those rigorous values throughout. Because all utility and
leakage effects are **monotone in σ**, re-labelling the ε axis leaves every ordering, retention %,
and AUC-drop unchanged.

## 6. Experimental Setup

**Datasets / payloads.**
- **LongMemEval (oracle)** [8]: real long-horizon conversational memory. Users = question
  haystacks, notes = dialogue turns, queries = questions, ground truth = evidence turns. The
  loader yields **500 users, 10,957 note embeddings, 479 queries with evidence.**
- **LongMemEval (`s`)** [8]: the full-haystack variant — same 500 users but **246,073 note
  embeddings** (~96% distractors), ~22× the oracle corpus — used for the at-scale generality
  check (§7.7).
- **LLM-distilled notes** (A-MEM/Mem0-style): dialogue turns distilled into memory notes by a
  local LLM (Ollama `qwen2.5:7b`). **500 users → 6,116 distilled notes**; a 100-user pilot →
  1,715 notes. This is the realistic agent-memory payload.
- **Synthetic / real-embedding controls**: 20-Newsgroups with TF-IDF→SVD and with real
  `all-MiniLM-L6-v2` embeddings, for controlled `N`/`K`/`d` sweeps and a clean topic-accuracy
  metric.

**Embeddings.** `all-MiniLM-L6-v2` (384-d) [7], PCA-reduced to the working dimension `d`
(PCA fit on notes, applied to queries). Buckets are `K` random-hyperplane anchors.

**Mechanism.** Faithful crypto path (`privacy_utils` quantize → Skellam → SecAgg-sum →
dequantize), `range_max = 1e6`. Vector-only release unless stated.

**Metrics.**
- *Utility*: evidence-recall@5 (LongMemEval), answer-recall@5 (distilled), topic-accuracy
  (controls); chance ≈ `topk/K`. We report **retention@ε = utility(ε)/utility(clean)**.
- *Leakage*: membership-inference ROC-AUC and **low-FPR TPR** (calibrated LiRA, §7.8) over all
  buckets and over the **low-count tail** (≤3 contributors), plus a reconstruction-decode
  extraction rate. 0.5 AUC / TPR≈FPR / chance-level decode = no leakage.

**Protocol.** **5 seeds (0–4)**, **final metric** (not peak), mean ± std (the calibrated LiRA of
§7.8 uses 3 seeds × 48 shadow releases). Members vs non-members are a same-distribution split of
the note pool (aggregated vs held-out), avoiding any train/test confound.

> **Status.** The **full `s`-variant (246k notes, ~96% distractors)** at-scale generality check
> (§7.7) and the **LLM-distilled-payload** results (§7.4) are complete, and the embedding-similarity
> MIA/extraction proxy of §7.2–7.4 is **upgraded to a calibrated offline-LiRA membership-inference
> attack and a reconstruction-decode extraction attack** (§7.8), which report the modern low-FPR
> TPR metric. The one remaining **[PENDING]** item is the *LLM-agent-prompting* form of
> MEXTRA/MRMMIA — a live agent querying a text store, a different release model than our centroid
> pool — which we scope as future work; the mechanism destroys the membership/reconstruction signal
> those attacks also rely on, so the endpoint is expected to hold.

## 7. Results

The tables below are regenerated directly from the released grid (`experiment_results/rerun_grid/`)
by the code cell at the end of this notebook, so they stay in sync with the artifacts. All numbers
are **5-seed means**; ε values are the rigorous Skellam-RDP labels (§5.4), where the exploration
"ε=8/ε=3" = **ε≈9.3 / 3.2** (vector-only).

### 7.1 Usable DP memory (LongMemEval oracle)

Retrieval survives DP in the tiny-d / coarse-K regime and degrades gracefully as memory fidelity
`K` grows (evidence-recall@5, d=32, separate release):

| K | chance | clean | ε≈9.3 (was ε=8) | ε≈3.2 (was ε=3) | retention@ε≈9.3 |
|---|---|---|---|---|---|
| **32** | 0.156 | 0.577 | **0.552** | 0.469 | **96%** |
| 64 | 0.078 | 0.469 | 0.418 | 0.308 | 89% |
| 128 | 0.039 | 0.385 | 0.298 | 0.162 | 77% |
| 256 | 0.020 | 0.321 | 0.204 | 0.098 | 64% |

At the recommended operating point (**K=32, d=32**) the private pool keeps **96%** of clean
evidence-recall at ε≈9.3 and **81%** at ε≈3.2, at **3.5×** chance. Utility is set by *effective
contributors per bucket*: coarse buckets pool more users, so DP is nearly free; fine buckets
starve, so noise bites. The d-sweep at fixed K=128 shows the same effect on the dimension axis:
retention 77% → 67% → 54% for d = 32 → 64 → 128.

### 7.2 The dimension/density crossover (real embeddings)

On **real `all-MiniLM-L6-v2`** embeddings (N=100, K=32), growing `d` **hurts both axes at once**:

| d | clean util | retention@ε≈9.3 | clean tail-AUC (leakage, K=1024) |
|---|---|---|---|
| **32** | 0.356 | **91%** | **0.944** |
| 64 | 0.359 | 87% | 0.970 |
| 128 | 0.358 | 78% | 0.985 |
| 384 | 0.318 | 56% | 0.989 |

Higher `d` means **more pre-DP leakage** (tail-AUC 0.94→0.99: high-d embeddings are more unique,
hence more re-identifiable) **and less DP utility-retention** (91%→56%: more coordinates to
noise). **Tiny-d is therefore Pareto-preferred** — the central design principle, and exactly the
regime agent-memory payloads occupy. (Figure: `figures/fig_d_crossover`.)

### 7.3 The leakage-drop endpoint (the positive result)

Membership inference on the **vulnerable low-count tail** collapses to chance under DP while bulk
utility holds (LongMemEval oracle; MIA tail-AUC, 0.5 = no leakage):

| K | tail % | clean tail-AUC | ε≈9.3 tail-AUC | ε≈3.2 tail-AUC |
|---|---|---|---|---|
| 512 | 1% | 0.873 | 0.584 | 0.588 |
| 1024 | 6% | **0.881** | **0.549** | 0.529 |
| 2048 | 17% | 0.872 | 0.515 | 0.509 |

Clean memory re-identifies tail members at **AUC ≈ 0.88** (near-certain on this real corpus; up
to **0.94–0.99** on the higher-fidelity synthetic/real-embedding configs of §7.2). DP drives it to
**≈0.51–0.55 (chance) at ε≈9.3**, and the extraction gap collapses in lockstep, while §7.1 utility
holds. This is the **positive, mechanism-grounded endpoint**: privacy is not asserted from the ε
label alone — the *actual attack* is measured to fail. Leakage rises with fidelity `K` pre-DP but
DP pins the attack at chance across `K`, **extending the usable-fidelity frontier**. (Figure:
`figures/fig_leakage_drop`.)

### 7.4 Realistic payload: LLM-distilled notes

The real agent-memory payload (distilled notes) **strengthens** the case. The distillation is a
fresh local-LLM run (Ollama `qwen2.5:7b`; 500 users → 6,116 notes, 100-user pilot → 1,715 notes),
so absolute values differ slightly from any single earlier run but the ordering is unchanged.

**Utility** (answer-recall@5, vector-only release) — density lifts DP retention:

| N (users) | K | clean | ε≈9.3 | retention |
|---|---|---|---|---|
| **500** | 32 | 0.598 | **0.573** | **96%** |
| 500 | 64 | 0.508 | 0.457 | 90% |
| 100 (pilot) | 32 | 0.638 | 0.448 | 70% |

At full scale (500 users, denser buckets) the distilled pool retains **96% at ε≈9.3**; the
100-user pilot retains 70% — **density, not scale per se, governs retention** (more contributors
per bucket ⇒ cheaper DP).

**Leakage** — distilled notes leak **more** than raw turns, yet DP kills both (full 500-user,
K=1024, tail-AUC):

| source | clean all-AUC | clean tail-AUC | ε≈9.3 tail-AUC |
|---|---|---|---|
| raw dialogue turns | 0.638 | 0.895 | 0.572 |
| **LLM-distilled notes** | **0.726** | **0.925** | 0.553 |

Distillation **concentrates identifying facts**, so distilled memory is *more* re-identifiable pre-DP
(all-AUC 0.64→0.73, tail 0.90→0.93) — but the private release still drives both to ≈0.55 at
ε≈9.3. The payload agents actually store is the one DP protects most decisively. (Figures:
`figures/fig_distilled_utility`, `figures/fig_distilled_leakage`.)

### 7.5 Accounting results

- **Discretisation is free.** At `range_max = 1e6` the discrete Skellam ε equals the Gaussian-RDP
  ε to 4+ decimals (surcharge < 1e-3); the integer/SecAgg quantisation costs no privacy.
- **Vector-only strictly dominates.** At matched σ the vector-only release gives **identical
  recall to the last decimal** as the two-channel design, at **ε 13.9 → 9.3** — because the count
  channel only ever cancelled under normalisation. One composition, ~1.5× tighter ε, same utility.

### 7.6 Robustness

Every number above is a **5-seed mean with the final (not peak) metric**. Re-running the full
grid at 5 seeds reproduced the 2-seed exploration headlines with **no sign flip** (e.g. K=32
retention 95%→96%; oracle tail-AUC 0.87→0.55 preserved), addressing the single-seed / peak-metric
pitfalls that inflate DP-FL results.

### 7.7 At-scale generality (LongMemEval `s`, 246k notes)

We repeat the utility and leakage measurements on the **full `s` variant** — the same 500 users but
**246,073 notes** (~96% distractors), ~22× the oracle corpus — the paper's at-scale generality
check. Both axes behave exactly as §8's density argument predicts.

**Utility is nearly free** (evidence-recall@5, d=32, 5-seed):

| K | chance | clean | ε≈9.3 | ε≈3.2 | retention@ε≈9.3 |
|---|---|---|---|---|---|
| **32** | 0.156 | 0.525 | 0.523 | 0.527 | **99%** |
| 64 | 0.078 | 0.405 | 0.404 | 0.397 | **100%** |
| 128 | 0.039 | 0.320 | 0.313 | 0.304 | **98%** |
| 256 | 0.020 | 0.271 | 0.262 | 0.233 | **97%** |

**The vulnerable tail vanishes.** At this density essentially all 500 users contribute to every
bucket, so the low-count tail (≤3 contributors) is **empty (0% of members)** across
K ∈ {512, 1024, 2048}, and membership inference is **already at chance without DP** (all-AUC ≈ 0.50):

| K | low-count tail % | clean all-AUC | ε≈9.3 all-AUC |
|---|---|---|---|
| 512 | 0.0% | 0.499 | 0.500 |
| 1024 | 0.0% | 0.512 | 0.507 |
| 2048 | 0.0% | 0.511 | 0.502 |

This is the honest at-scale reading promised in §8: the dramatic leakage-*drop* is a sparse-tail
phenomenon (§7.3), whereas the dense at-scale regime is **safe on both axes for free** — averaging
over hundreds of contributors per bucket already destroys the membership signal, and DP costs almost
no utility. The at-scale run therefore **confirms and bounds** the headline rather than extending it.

### 7.8 Calibrated attack: LiRA membership inference + extraction

The similarity scores in §7.2–7.4 embody the *measurement principle* of MEXTRA/MRMMIA but are a
weak, uncalibrated proxy. We now run the **modern MIA standard** against the released pool: an
**offline LiRA** (Carlini et al., S&P 2022) that, for each candidate note, estimates its
membership-score distribution when it is *aggregated into* vs *held out of* the pool from many
**shadow releases** (cheap here — our aggregation is numpy, not model training) and tests with the
per-target likelihood ratio. We report **ROC-AUC and the low-FPR TPR** (the operationally
meaningful metric that AUC-only proxies hide), plus a **reconstruction-decode extraction** attack
(MEXTRA analog): decode each released centroid to its nearest note and score a hit when the top-1
decoded note is a true in-bucket member. Fits use a shadow split disjoint from the evaluation
shadows (no train-on-test bias).

**The clean release is far more leaky than the proxy revealed — and DP still defeats the strong
attack** (LongMemEval oracle, d=32, 3 seeds × 48 shadow releases):

| K | release | AUC | TPR@1%FPR | TPR@0.1%FPR | decode-extract |
|---|---|---|---|---|---|
| 1024 | clean | 0.994 | **0.898** | 0.778 | 0.854 |
| 1024 | ε≈9.3 | 0.534 | **0.011** | 0.001 | 0.078 |
| 1024 | ε≈3.2 | 0.505 | 0.011 | 0.001 | 0.018 |

Calibration exposes what cosine-AUC missed: on the clean pool an adversary re-identifies members at
**≈90% true-positive rate at a 1% false-positive budget** and reconstructs the correct in-bucket
note for **85% of centroids** — the untreated shared memory is almost fully de-anonymising. The
Skellam release drives the same attack to chance: **TPR@1%FPR 0.90→0.01** (the FPR floor) and
**extraction 0.85→0.08→≈0**. The effect holds across the fidelity sweep — clean leakage rises with
`K` exactly as §7.3 predicted, while DP pins every cell at chance:

| K | clean TPR@1%FPR | ε≈9.3 TPR@1%FPR | clean decode | ε≈9.3 decode |
|---|---|---|---|---|
| 512 | 0.802 | 0.012 | 0.809 | 0.177 |
| 1024 | 0.898 | 0.011 | 0.854 | 0.078 |
| 2048 | 0.916 | 0.012 | 0.895 | 0.039 |

On **real `all-MiniLM-L6-v2` embeddings** the dimension axis behaves identically: clean TPR@1%FPR
climbs **0.91→0.98** and decode **0.80→0.97** from d=32→384 (higher-`d` memory is more
re-identifiable), and DP collapses both to chance at ε≈9.3 — the §7.2 crossover, now with a
calibrated attack. (We headline the low-FPR TPR because AUC becomes an unstable estimator at the
largest σ, where it can tick up to ≈0.59 even as TPR@1%FPR stays at the ≈1% floor; the attack is
defeated operationally regardless.) This **upgrades the endpoint from a proxy to a calibrated
attack**, leaving only the *LLM-agent-prompting* form of MEXTRA/MRMMIA (a live agent querying a
text store — a different release model than our centroid pool) as future work.

## 8. Discussion and Limitations

**The leakage-drop headline is a sparse-regime property — stated honestly.** The dramatic
0.9→0.5 tail-AUC drop lives in the **low-count tail**; bulk-bucket AUC is only 0.53–0.59 pre-DP
because averaging already protects dense buckets. Density governs both axes: a very dense,
coarse-`K` regime makes DP nearly free **and** has little low-count tail to begin with (so a
smaller headline drop). We therefore frame the result as *"DP provably protects the vulnerable
tail that a sum mechanism most exposes,"* not as an unconditional leakage collapse. The at-scale
`s`-variant (§7.7) **confirms this directly**: with 246k notes it sits in the dense regime, where
the low-count tail is **empty** and clean membership-inference AUC is **already ≈0.50**, so DP is
nearly free on utility (97–100% retention) and there is little tail leakage left to drop — reported
as such, not overclaimed.

**Attack strength.** §7.2–7.4 use an embedding-similarity membership/extraction proxy; §7.8
upgrades it to a **calibrated offline-LiRA** MIA (reported at low-FPR TPR, the modern standard) and
a **reconstruction-decode extraction** attack. Calibration reveals the clean pool is *more* leaky
than the proxy showed — TPR@1%FPR ≈ 0.90 and top-1 note reconstruction for ≈85% of centroids — yet
the Skellam release still drives the strong attack to chance (TPR@1%FPR → ≈0.01, extraction → ≈0),
so the endpoint **strengthens** under a better attack rather than weakening. The remaining
**[PENDING]** item is the *LLM-agent-prompting* form of MEXTRA/MRMMIA — a live agent querying a
text store, a different release model than our centroid pool; we expect the endpoint to hold since
the mechanism destroys the same membership/reconstruction signal, but the absolute rates may shift.

**Scope.** Utility is retrieval fidelity of the pooled memory (bucket routing), not end-to-end
downstream QA; extending to generated-answer quality is future work. Retrieval is bucket-granular,
not exact-note ranking. `K`/`d`/clip bounds are set from data percentiles, not tuned per user.

**Non-novel components, explicitly.** The dimension dependence of DP is classical [19, 20]; we
claim only its *coupled* manifestation across leakage and utility in agent memory. The SecAgg +
Skellam path is prior work [2, 5, 6]; the novelty is payload + endpoint.

## 9. Conclusion

Shared LLM-agent memory can be made **useful and provably private simultaneously**. By aggregating
per-user bucketed memory embeddings under Secure Aggregation and the Skellam mechanism, the pooled
memory carries a curator-free central-DP guarantee, retains **96–97% of clean retrieval utility at
ε≈9.3** in the tiny-d / coarse-K regime agent payloads occupy, and drives worst-case membership
inference on vulnerable members from **near-certain to chance**. The dimension/density crossover
makes tiny-d a joint optimum, the discrete mechanism is privacy-free at our resolution, a
vector-only release strictly dominates, and — reassuringly — the realistic distilled payload,
which leaks *more* in the clear, is protected *more* decisively by DP. The remaining at-scale and
full-attack runs are expected to confirm, not overturn, these findings.

## 10. Reproducibility

Active code lives in **`scripts/agentmem/`** (see `scripts/README.md`); the pooled crypto is
`qpriviot_fl/privacy_utils.py`.

- **Utility**: `_longmemeval_probe.py` (real), `_agentmem_probe.py` (controls),
  `_longmemeval_distilled_utility.py` (distilled).
- **Leakage (proxy)**: `_longmemeval_leakage.py`, `_agentmem_leakage.py`,
  `_longmemeval_distilled_analysis.py` (raw-vs-distilled).
- **Leakage (calibrated, §7.8)**: `_agentmem_lira.py` — offline-LiRA MIA (AUC + low-FPR TPR) and
  reconstruction-decode extraction, via shadow releases.
- **Accounting**: `_skellam_accounting.py` (Skellam-RDP ε; discretisation-free check).
- **Distillation**: `_longmemeval_distill.py` turns LongMemEval turns into A-MEM/Mem0-style notes
  via a local Ollama model (`qwen2.5:7b`); it checkpoints incrementally.
- **Drivers → tables/figures**: `scripts/shell/rerun_grid.sh` runs the 33-config, 5-seed grid into
  `experiment_results/rerun_grid/*.json`; `scripts/shell/rerun_svariant.sh` runs the at-scale
  `s`-variant (§7.7) into `experiment_results/rerun_grid_s/*.json`; `scripts/shell/rerun_lira.sh`
  runs the calibrated-attack grid (§7.8) into `experiment_results/lira/*.json`; `rerun_tables.py`
  and `rerun_figures.py` regenerate `SUMMARY_TABLE.md` and `figures/fig_*`. Each script has an
  additive `--json` dump; the crypto path is byte-identical to the released `privacy_utils`.

Run: `bash scripts/shell/rerun_grid.sh && python scripts/agentmem/rerun_tables.py && python scripts/agentmem/rerun_figures.py`
(and `bash scripts/shell/rerun_svariant.sh` for §7.7, `bash scripts/shell/rerun_lira.sh` for §7.8).

## References

*Note: arXiv identifiers for the 2026 agent-memory works are taken from the project's research
notes and must be verified against the published record before submission.*

[1] McMahan et al. *Communication-Efficient Learning of Deep Networks from Decentralized Data.* AISTATS 2017.
[2] Bonawitz et al. *Practical Secure Aggregation for Privacy-Preserving Machine Learning.* CCS 2017.
[3] Abadi et al. *Deep Learning with Differential Privacy.* CCS 2016.
[4] Mironov. *Rényi Differential Privacy.* CSF 2017.
[5] Agarwal, Kairouz, Liu. *The Skellam Mechanism for Differentially Private Federated Learning.* NeurIPS 2021 (arXiv:2110.04995).
[6] Kairouz et al. *The Distributed Discrete Gaussian Mechanism for Federated Learning with Secure Aggregation.* ICML 2021.
[7] Reimers, Gurevych. *Sentence-BERT.* EMNLP 2019 (all-MiniLM-L6-v2).
[8] Wu et al. *LongMemEval: Benchmarking Chat Assistants on Long-Term Interactive Memory.* 2024.
[9] *A-MEM: Agentic Memory for LLM Agents.* 2024.
[10] *Mem0: Building Production-Ready AI Agents with Scalable Long-Term Memory.* 2024.
[11] *MEXTRA: Memory Extraction Attacks against LLM Agents.* arXiv:2502.13172.
[12] *MRMMIA: Membership Inference against Agent Memory.* arXiv:2605.27825.
[13] *Collaborative Memory.* arXiv:2505.18279.
[14] *MemPrivacy: On-Device Memory Masking.* arXiv:2605.09530.
[15] *Differentially Private Datastore Generation for Retrieval.* arXiv:2606.01413.
[16] *POPri: User-Level DP Preference Aggregation under Secure Aggregation.* arXiv:2504.16438.
[17] *Fed-SE: Federated Skill Embeddings for Agents.* arXiv:2512.08870.
[18] *Survey on Privacy in Federated LLM-Agent Adaptation.* arXiv:2604.16548.
[19] Chen et al. *The Effect of Dimensionality on Differentially Private Deep Learning.* ICML 2022 (arXiv:2203.03761).
[20] Bassily, Smith, Thakurta. *Private Empirical Risk Minimization.* FOCS 2014.

In [ ]:
# Regenerate the results tables + figures inline from the released grid, so the paper
# numbers stay in sync with experiment_results/rerun_grid/. Run from the repo root.
from pathlib import Path
import json, textwrap

REPO = Path.cwd()
while REPO.name and not (REPO / "experiment_results" / "rerun_grid").exists() and REPO != REPO.parent:
    REPO = REPO.parent
GRID = REPO / "experiment_results" / "rerun_grid"

summary = GRID / "SUMMARY_TABLE.md"
if summary.exists():
    print(summary.read_text())
else:
    print("Run: bash scripts/shell/rerun_grid.sh && python scripts/agentmem/rerun_tables.py")

# Display the four headline figures if present
try:
    from IPython.display import Image, display
    for name in ["fig_utility_vs_eps", "fig_leakage_drop", "fig_d_crossover",
                 "fig_distilled_utility", "fig_distilled_leakage"]:
        p = REPO / "figures" / f"{name}.png"
        if p.exists():
            print(f"\\n=== {name} ===")
            display(Image(filename=str(p)))
except Exception as e:
    print("(figure display skipped:", e, ")")